In [ ]:
from trl import DPOTrainer, DPOConfig
from transformers import AutoTokenizer,  AutoModelForCausalLM, TrainingArguments
from peft import PeftModel
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import torch

In [ ]:
base_model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
import zipfile
import os
zip_path = "/content/tinyllama-instruction.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

In [ ]:
model_path = "/content/checkpoint-3"

In [ ]:
import pip
import pkg_resources
try:
    # Check for torchao version and upgrade if necessary
    torchao_version = pkg_resources.get_distribution("torchao").version
    if pkg_resources.parse_version(torchao_version) < pkg_resources.parse_version("0.16.0"):
        print(f"Upgrading torchao from {torchao_version} to a version >=0.16.0...")
        !pip install torchao --upgrade --quiet
except pkg_resources.DistributionNotFound:
    print("torchao not found, installing...")
    !pip install torchao --quiet

instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

In [ ]:
!pip install -U trl

In [ ]:
!pip install -U bitsandbytes

In [ ]:
dataset = load_dataset("csv", data_files="/content/pharma_preference_data.csv")["train"]

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [ ]:
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=quantization_config,
    device_map="auto"
)

In [ ]:
instruction_checkpoint = "/content/checkpoint-3"

In [ ]:
model = PeftModel.from_pretrained(model, instruction_checkpoint)

In [ ]:
# model = model.merge_and_unload()

In [ ]:
pref_model_lora = get_peft_model(model, lora_config)

In [ ]:
dpo_args = DPOConfig(
    output_dir="./tinyllama-preference-alignment",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    beta=0.1,
    report_to=[],
    logging_dir=None,
    loss_type="sigmoid",
    remove_unused_columns=False
)

In [ ]:
trainer = DPOTrainer(
    model=pref_model_lora,
    ref_model=None,
    args=dpo_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

In [ ]:
question = "how does Metformin works in the human and does it have benefits for diabetes patient?"

In [ ]:
model_path = "/content/tinyllama-preference-alignment/checkpoint-1"

In [ ]:
zip_path = "/content/tinyllama-non-instruction.zip"


with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

In [ ]:
model_path = "/content/checkpoint-5"
non_instruction_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=quantization_config,
    device_map="auto"
)

In [ ]:
inputs = tokenizer(question, return_tensors="pt").to("cuda")

In [ ]:
outputs = non_instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
model_path = "/content/checkpoint-3"
instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

In [ ]:
inputs = tokenizer(question, return_tensors="pt").to("cuda")

In [ ]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
model_path = "/content/tinyllama-preference-alignment/checkpoint-1"

In [ ]:
base_model_for_inference = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=quantization_config,
    device_map="auto"
)

preference_aligned_model = PeftModel.from_pretrained(base_model_for_inference, model_path)

In [ ]:
preference_aligned_model.to("cuda")

In [ ]:
outputs = preference_aligned_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))